In [ ]:
# Kaggle setup (does nothing outside Kaggle). Needs: GPU on, Internet on, competition data as input.
# The repo stays private until the deadline, so it is cloned with a GitHub token saved as a
# Kaggle Secret called GITHUB_TOKEN (Add-ons -> Secrets; fine-grained token, read-only, this repo only).
import os, subprocess
REPO = "polluvl/hw2-video-detection"   # <- set once, e.g. "polluvl/hw2-video-detection"
if os.path.isdir("/kaggle/working") and not os.path.isdir("/kaggle/working/repo"):
    try:
        from kaggle_secrets import UserSecretsClient
        auth = UserSecretsClient().get_secret("GITHUB_TOKEN") + "@"
    except Exception:
        auth = ""  # no secret -> works once the repo is public
    r = subprocess.run(["git", "clone", "-q", f"https://{auth}github.com/{REPO}.git", "/kaggle/working/repo"],
                       capture_output=True)
    if r.returncode != 0:  # do not print the command: it contains the token
        raise RuntimeError("git clone failed: check REPO and the GITHUB_TOKEN secret")
    # remove the token from the saved clone (outputs of /kaggle/working can be shared)
    subprocess.run(["git", "-C", "/kaggle/working/repo", "remote", "set-url", "origin",
                    f"https://github.com/{REPO}.git"], check=True)
    subprocess.run(["pip", "install", "-q", "-r", "/kaggle/working/repo/requirements.txt"], check=True)
if os.path.isdir("/kaggle/working/repo/notebooks"):
    os.chdir("/kaggle/working/repo/notebooks")

# 06 · Final pipeline
Starts from the best detector of 02–05 (by our val mAP) and changes **one thing at a time**,
keeping two groups apart as the assignment asks:
- **A. data / model changes** (need retraining): more training frames; a bigger model
- **B. inference changes** (same weights): test resolution, flip TTA, tiling, temporal rescoring,
  ensembling with WBF

All B experiments are evaluated on **all** val frames (step 1), because temporal rescoring needs
consecutive frames. These numbers are therefore not directly comparable to the step-5 numbers
of 02–05; the first row (`pipe_base`) is the reference for this table.

In [ ]:
import ast
import sys
import time
from pathlib import Path

import pandas as pd
import torch
from ultralytics import YOLO

sys.path.insert(0, str(Path("../src").resolve()))
sys.path.insert(0, str(Path("src").resolve()))
from common import (SMOKE, evaluate, evaluate_per_clip, find_data_root,
                    list_frames, list_test_frames, load_preds, log_result,
                    save_preds, seed_everything, work_dir, write_submission)
from inference import merge_wbf, predict_tiled, temporal_rescore
from yolo_utils import export_yolo_split, predict_ultralytics, write_data_yaml

DATA = find_data_root()
OUT = work_dir()
seed_everything(42)
DEVICE = 0 if torch.cuda.is_available() else "cpu"

RUN_DATA_EXP = True    # A1: retrain best config on 3x more frames (~2-3 GPU h)
RUN_MODEL_EXP = True   # A2: YOLOv8m instead of YOLOv8s (~3 GPU h)

EPOCHS = 1 if SMOKE else 30
results = pd.read_csv(OUT / "results.csv")
yolo_runs = results[results.experiment.str.startswith("yolo")].sort_values("mAP")
BEST = yolo_runs.experiment.iloc[-1]
BEST_SPEC = ast.literal_eval(yolo_runs.config.iloc[-1])  # the dict logged by 04 (imgsz, box, ...)
IMGSZ = BEST_SPEC["imgsz"]
print("starting point:", BEST, BEST_SPEC)
results[["experiment", "family", "mAP", "AP50", "AP_small", "ms_per_img"]].sort_values("mAP")

## A. Data and model changes (evaluated on the same step-5 val frames as 02–05)

In [ ]:
val5 = list_frames(DATA / "val", step=1 if SMOKE else 5)
base_model_file = "yolov8s-p2.yaml" if "p2" in BEST else "yolov8s.pt"


def train_yolo(name, model_file, train_step, extra=None, weights=None):
    ds = OUT / f"yolo_ds_step{train_step}"
    if not (ds / "data.yaml").exists():
        export_yolo_split(list_frames(DATA / "train", step=train_step), ds / "train")
        export_yolo_split(val5, ds / "val")
        write_data_yaml(ds / "data.yaml", ds / "train", ds / "val")
    model = YOLO(model_file)
    if weights:
        model.load(weights)
    args = dict(data=str(ds / "data.yaml"), epochs=EPOCHS, imgsz=IMGSZ, batch=2 if SMOKE else 8,
                workers=2, seed=42, deterministic=True, device=DEVICE, project=str(OUT / "ultralytics"),
                exist_ok=True, cos_lr=True, close_mosaic=5, max_det=500, plots=not SMOKE, name=name)
    args.update({k: v for k, v in BEST_SPEC.items() if k not in ("model", "weights", "imgsz")})
    args.update(extra or {})
    model.train(**args)
    best = YOLO(str(OUT / "ultralytics" / name / "weights" / "best.pt"))
    preds, ms = predict_ultralytics(best, val5, imgsz=IMGSZ)
    save_preds(preds, OUT / f"preds_val_{name}.npz")
    m = evaluate(val5, preds)
    log_result(name, m, {"family": "pipeline (data/model)", "ms_per_img": round(ms, 1),
                         "config": f"{model_file}, train_step={train_step}, {extra or {}}"})
    print(name, m)
    return m


p2_weights = None if SMOKE else ("yolov8s.pt" if "p2" in BEST else None)
if SMOKE:
    base_model_file = base_model_file.replace(".pt", ".yaml")
if RUN_DATA_EXP:  # data change only: every 3rd frame instead of every 10th
    train_yolo("pipe_data_step3", base_model_file, 1 if SMOKE else 3, weights=p2_weights)
if RUN_MODEL_EXP:  # model change only: same data as BEST, bigger network
    m_file = base_model_file.replace("yolov8s", "yolov8m")
    train_yolo("pipe_model_m", m_file, 1 if SMOKE else 10,
               weights=None if SMOKE else ("yolov8m.pt" if "p2" in BEST else None))

results = pd.read_csv(OUT / "results.csv")
cand = results.set_index("experiment").loc[[r for r in [BEST, "pipe_data_step3", "pipe_model_m"]
                                            if r in set(results.experiment)]].reset_index()
FINAL_RUN = cand.sort_values("mAP", ascending=False, kind="stable").experiment.iloc[0]  # ties -> BEST
print(cand[["experiment", "mAP", "AP50", "AP_small", "config"]])
print("model used for the inference experiments:", FINAL_RUN)

## B. Inference strategies (same weights), all val frames

In [ ]:
val_all = list_frames(DATA / "val", step=1)
model = YOLO(str(OUT / "ultralytics" / FINAL_RUN / "weights" / "best.pt"))
pipe = {}


def record(name, preds, ms, note):
    save_preds(preds, OUT / f"preds_valall_{name}.npz")
    m = evaluate(val_all, preds)
    log_result(name, m, {"family": "pipeline (inference)", "ms_per_img": round(ms, 1), "config": note})
    pipe[name] = preds
    print(f"{name:22s} mAP {m['mAP']:.4f} AP50 {m['AP50']:.4f} AP_s {m['AP_small']:.4f} ({ms:.0f} ms/img)")
    return m


p, ms = predict_ultralytics(model, val_all, imgsz=IMGSZ)
record("pipe_base", p, ms, f"{FINAL_RUN}, imgsz={IMGSZ}")

big = int(IMGSZ * 1.25 // 32 * 32)
p, ms = predict_ultralytics(model, val_all, imgsz=big)
record("pipe_imgsz_up", p, ms, f"test imgsz={big}")

p, ms = predict_ultralytics(model, val_all, imgsz=IMGSZ, augment=True)
record("pipe_tta", p, ms, "Ultralytics TTA (flip + scales)")

t0 = time.time()
p = predict_tiled(model, val_all, imgsz=IMGSZ, grid=(2, 2), overlap=0.2, merge="nms")
record("pipe_tiles", p, 1000 * (time.time() - t0) / len(val_all), "full image + 2x2 tiles, NMS merge")

### Temporal rescoring (bonus) on top of the best single-pass strategy
Small grid over the matching IoU and the smoothing weight, on val only.

In [ ]:
pipe_res = pd.read_csv(OUT / "results.csv")
# on a tie keep the earlier = cheaper strategy (stable sort, descending)
pipe_res = pipe_res[pipe_res.experiment.isin(pipe)].sort_values("mAP", ascending=False, kind="stable")
BEST_INF = pipe_res.experiment.iloc[0]
print("best inference strategy so far:", BEST_INF)

grid = []
for iou_thr in [0.2, 0.3, 0.5]:
    for weight in [0.3, 0.5, 0.7]:
        for causal in [False, True]:
            p = temporal_rescore(val_all, pipe[BEST_INF], iou_thr=iou_thr, weight=weight, causal=causal)
            m = evaluate(val_all, p)
            grid.append({"iou_thr": iou_thr, "weight": weight, "causal": causal, **m})
grid = pd.DataFrame(grid).sort_values("mAP", ascending=False)
print(grid[["iou_thr", "weight", "causal", "mAP", "AP50", "AP_small"]].head(8))

best_t = grid.iloc[0]
TEMPORAL = dict(iou_thr=float(best_t.iou_thr), weight=float(best_t.weight), causal=bool(best_t.causal))
p = temporal_rescore(val_all, pipe[BEST_INF], **TEMPORAL)
record("pipe_temporal", p, 0, f"{BEST_INF} + temporal {TEMPORAL}")
print("per-clip mAP before / after temporal rescoring:")
pd.merge(evaluate_per_clip(val_all, pipe[BEST_INF]), evaluate_per_clip(val_all, p),
         on=["clip", "n_frames"], suffixes=("_before", "_after"))

### Ensemble (WBF) of the two best different models, if both exist
(only meaningful if their errors differ; checked on val before using it)

In [ ]:
others = [r for r in [BEST, "pipe_data_step3", "pipe_model_m"]
          if r != FINAL_RUN and (OUT / "ultralytics" / r / "weights" / "best.pt").exists()]
if others:
    second = YOLO(str(OUT / "ultralytics" / others[0] / "weights" / "best.pt"))
    p2, _ = predict_ultralytics(second, val_all, imgsz=IMGSZ)
    ens = {r.image_key: merge_wbf([pipe["pipe_base"][r.image_key], p2[r.image_key]], r.width, r.height)
           for r in val_all.itertuples()}
    record("pipe_wbf_2models", ens, 0, f"WBF({FINAL_RUN}, {others[0]})")

## Final choice and test submission
The chosen combination is decided ONLY by val mAP above (not by the public leaderboard).

In [ ]:
pipe_res = pd.read_csv(OUT / "results.csv")
pipe_res = pipe_res[pipe_res.experiment.str.startswith("pipe_") & (pipe_res.family == "pipeline (inference)")]
print(pipe_res.sort_values("mAP", ascending=False)[["experiment", "mAP", "AP50", "AP_small", "ms_per_img", "config"]])

USE_TEMPORAL = pipe_res.set_index("experiment").mAP.get("pipe_temporal", 0) > \
    pipe_res.set_index("experiment").mAP.get(BEST_INF, 0)
print(f"final: {FINAL_RUN} + {BEST_INF}" + (f" + temporal {TEMPORAL}" if USE_TEMPORAL else ""))

def run_strategy(frames, strategy):
    if strategy == "pipe_imgsz_up":
        return predict_ultralytics(model, frames, imgsz=big)[0]
    if strategy == "pipe_tta":
        return predict_ultralytics(model, frames, imgsz=IMGSZ, augment=True)[0]
    if strategy == "pipe_tiles":
        return predict_tiled(model, frames, imgsz=IMGSZ, grid=(2, 2), overlap=0.2, merge="nms")
    return predict_ultralytics(model, frames, imgsz=IMGSZ)[0]


test_frames = list_test_frames(DATA)
test_preds = run_strategy(test_frames, BEST_INF)
if USE_TEMPORAL:
    test_preds = temporal_rescore(test_frames, test_preds, **TEMPORAL)
save_preds(test_preds, OUT / "preds_test_final.npz")
sub = write_submission(test_frames, test_preds, OUT / "submission.csv")

# sanity checks on the file format
sample = pd.read_csv(DATA / "sample_submission.csv")
assert len(sub) == len(sample) and set(sub.image_id) == set(sample.image_id)
n_boxes = sub.PredictionString.fillna("").str.split().str.len().fillna(0) / 6
assert (n_boxes == n_boxes.round()).all() and n_boxes.max() <= 500
print(f"submission.csv: {len(sub)} rows, {n_boxes.mean():.0f} boxes per image on average")
sub.head()

## Summary of all models (val, step 5 unless stated)

In [ ]:
summary = pd.read_csv(OUT / "results.csv")
summary[["experiment", "family", "mAP", "AP50", "AP_small", "AP_medium", "AP_large", "ms_per_img", "train_min"]]